## Inspect l1 Logistic Regression Classifier

Imports

In [1]:
import os
import json
import argparse
import joblib
import numpy as np
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from sklearn.model_selection import cross_validate
from transformers import AutoConfig

/home/dzur/ai-projects/H-Neurons-Auto/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Import data

In [2]:
def load_data(ids_path, ans_acts_dir, other_acts_dir=None, mode="1-vs-1"):
    """
    Flexible data loader.
    1-vs-1: False Answer Tokens (Label 1) vs True Answer Tokens (Label 0).
    3-vs-1: False Answer Tokens (Label 1) vs (True Ans + True Other + False Other) (Label 0).
    """
    with open(ids_path, "r") as f:
        id_map = json.load(f)
    
    X, y = [], []

    # 1. Load False Answer Tokens -> Always Label 1 (Positive)
    for qid in tqdm(id_map["f"], desc="Loading False Ans (Label 1)"):
        path = os.path.join(ans_acts_dir, f"act_{qid}.npy")
        if os.path.exists(path):
            X.append(np.load(path).flatten())
            y.append(1)

    # 2. Load True Answer Tokens -> Always Label 0 (Negative)
    for qid in tqdm(id_map["t"], desc="Loading True Ans (Label 0)"):
        path = os.path.join(ans_acts_dir, f"act_{qid}.npy")
        if os.path.exists(path):
            X.append(np.load(path).flatten())
            y.append(0)

    # 3. Load Other Tokens if 3-vs-1 mode is enabled
    if mode == "3-vs-1":
        if not other_acts_dir:
            raise ValueError("train_other_acts directory is required for 3-vs-1 mode.")
        
        for label_key in ["t", "f"]:
            for qid in tqdm(id_map[label_key], desc=f"Loading Other Tokens - {label_key} (Label 0)"):
                path = os.path.join(other_acts_dir, f"act_{qid}.npy")
                if os.path.exists(path):
                    X.append(np.load(path).flatten())
                    y.append(0)

    return np.array(X), np.array(y)



In [3]:
X_train, y_train = load_data(
    ids_path="/home/dzur/ai-projects/H-Neurons-Auto/data/modernbert-path-examples/Gemma3-4b/train/Gemma3_4b_train_qids.json",
    ans_acts_dir="/home/dzur/ai-projects/H-Neurons-Auto/data/modernbert-path-examples/Gemma3-4b/train/answer_tokens",
    other_acts_dir="/home/dzur/ai-projects/H-Neurons-Auto/data/modernbert-path-examples/Gemma3-4b/train/all_except_answer_tokens",
    mode="3-vs-1"
)

Loading Other Tokens - f (Label 0): 100%|██████████| 1000/1000 [00:02<00:00, 408.80it/s]


Inspect

In [4]:
print(X_train.shape, y_train.shape)

(4000, 348160) (4000,)


In [7]:
print(X_train[1])

[3.6163330e-03 7.5988770e-03 3.4637451e-03 ... 4.3106079e-04 9.8228455e-05
 1.1901855e-03]


In [8]:
print(y_train[1])

1
